## Web scraping from UK petitions website

In [28]:
# Importing libraries
from bs4 import BeautifulSoup
from IPython.display import HTML
import requests
import pandas as pd
import time

pd.set_option('display.max_rows', None)

NameError: name 'df' is not defined

In [26]:
# Connection to website
url = 'https://petition.parliament.uk/petitions?state=awaiting_debate'
headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"}
soup = BeautifulSoup(requests.get(url).text, 'html.parser')

# Getting number of pages the petitions are spread over
page_count = soup.find('span', class_='page-count')

if page_count:
    total_pages = int(page_count.get_text().split('of')[-1].strip())
else:
    total_pages = 1

all_petitions_list = []
base_url = 'https://petition.parliament.uk/petitions?page={}&state=awaiting_debate'

for page in range(1, total_pages + 1):
    print(f"Fetching page {page}...")
    soup = BeautifulSoup(requests.get(base_url.format(page)).text, 'html.parser')
    
    for petition in soup.find_all('li', class_='petition-item'):
        link = petition.find('h2').find('a')
        all_petitions_list.append({
            'petition_id': link['href'].split('/')[-1],
            'petition_title': link.get_text(strip=True),
            'url': 'https://petition.parliament.uk' + link['href'],
        })
    
    time.sleep(1)

all_petitions = pd.DataFrame(all_petitions_list)

Fetching page 1...


In [18]:
all_petitions.shape

(15, 3)

In [20]:
# Connection to website
url = 'https://petition.parliament.uk/petitions/706513.json'

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36"}

petitions_counts = []
total_petitions = len(all_petitions)

for idx, petition in all_petitions.iterrows():
    print(f"Progress: {idx+1}/{total_petitions} ({(idx+1)/total_petitions*100:.1f}%) - {petition['petition_title'][:50]}...")

    try:
        data = requests.get(f"{petition['url']}.json", headers=headers).json()
        attributes = data['data']['attributes']
        debate_threshold_reached = attributes.get('scheduled_debate_date', None)

        for constituency in attributes['signatures_by_constituency']:
            petitions_counts.append({
                'petition_id': petition['petition_id'],
                'petition_title': petition['petition_title'],
                'scheduled_debate_date': debate_threshold_reached,
                'constituency_name': constituency['name'],
                'signature_count': constituency['signature_count']
            })
    
        time.sleep(1)
        
    except Exception as e:
        print(f" Error: {e}")
        continue

petitions_counts_df = pd.DataFrame(petitions_counts)

Progress: 1/15 (6.7%) - Urgently fulfil humanitarian obligations to Gaza...
Progress: 2/15 (13.3%) - Every school & college to be obliged to have an ev...
Progress: 3/15 (20.0%) - Withdraw the Children's Wellbeing and Schools Bill...
Progress: 4/15 (26.7%) - Do not introduce Digital ID cards...
Progress: 5/15 (33.3%) - Repeal the Online Safety Act...
Progress: 6/15 (40.0%) - Extend free bus travel for people over 60 in Engla...
Progress: 7/15 (46.7%) - Mandatory collection and publication of certain ch...
Progress: 8/15 (53.3%) - Call an immediate general election...
Progress: 9/15 (60.0%) - Introduce offshore detention/mass deportation for ...
Progress: 10/15 (66.7%) - Introduce Licensing and Regulation for Dog and Cat...
Progress: 11/15 (73.3%) - Reduce the school week to four days a week...
Progress: 12/15 (80.0%) - Make Play and Continuous Provision statutory in En...
Progress: 13/15 (86.7%) - Reduce the maximum noise level for consumer firewo...
Progress: 14/15 (93.3%) - Limit the

In [7]:
petitions_counts_df.to_csv('Data/petitions.csv')

,name,ons_code,mp,signature_count
0,Aldershot,E14001063,Alex Baker MP,199
1,Aldridge-Brownhills,E14001064,Rt Hon Wendy Morton MP,83
2,Altrincham and Sale West,E14001065,Mr Connor Rand MP,239
3,Amber Valley,E14001066,Linsey Farnsworth MP,88
4,Arundel and South Downs,E14001067,Andrew Griffith MP,192
...,...,...,...,...
645,Swansea West,W07000108,Torsten Bell MP,88
646,Torfaen,W07000109,Rt Hon Nick Thomas-Symonds MP,172
647,Vale of Glamorgan,W07000110,Kanishka Narayan MP,89
648,Wrexham,W07000111,Andrew Ranger MP,638


In [112]:
# Export files
df.to_csv('Data/706513 petition data.csv')